# Survival-metric choice — which TGI metric predicts OS is a model-selection axis

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only; no individual prognosis, no therapy ranking. Executed in CI (nbmake).

A composed forecast is `TGI model → on-treatment metric → survival link → OS`. Onkos already treats *which TGI model* and *which survival structure* (Weibull vs Cox) as model-selection axes; the **bridge metric** was a silent constant (the week-8 change). This notebook makes it swappable and adds the tail-sensitive **growth-rate constant k_g** link — completing the v0.24 finding that a week-8 surrogate is nearly blind to the regrowth tail. The result: the metric choice can **invert** which model looks better. See `docs/specs/research/survival-metric-choice.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 260, 521)
KG = 'survival_link.nsclc_os_growth_rate'
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The default (week-8) vs the growth-rate (k_g) metric

The default OS link reads the early week-8 change; the k_g link (non-default, opt-in via `survival_link=`) reads the post-nadir regrowth rate. Same TGI models, two bridge metrics.

In [ ]:
models = {
    'Claret (phenom.)':   'resistance.claret_2009.tgi',
    'two-pop (mechanistic)': 'resistance.nsclc_first_line.two_population',
    'Norton-Simon (CR)':  'drug_effect.norton_simon.nsclc',
    'Wang biexp':         'tgi_metrics.wang_2009.biexponential',
}
print(f"{'model':<24}{'week8 chg':>10}{'OS week8':>10}{'k_g':>9}{'OS k_g':>9}")
rows = {}
for name, rid in models.items():
    w = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t)
    g = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t, survival_link=KG)
    kg = w.metrics['tumor_growth_rate_kg']
    rows[name] = (w.median_os, g.median_os)
    print(f"{name:<24}{w.metrics['week8_relative_change']:>+10.2f}{w.median_os:>10.0f}"
          f"{(f'{kg:.3f}' if np.isfinite(kg) else 'none'):>9}{g.median_os:>9.0f}")

## Two inversions

1. **Resistance-model ranking flips.** Under week-8 the mechanistic model looks better (deeper early shrinkage); under k_g it looks worse (faster regrowth). The v0.24 tail divergence becomes decisive — and points the other way.
2. **The complete responder is undervalued by the surrogate.** Norton-Simon eradicates (no regrowth, k_g undefined → baseline hazard). Week-8 scores it mediocre; k_g makes it the longest survivor — an early-shrinkage endpoint penalizes a slow-but-complete responder.

In [ ]:
# Inversion 1: resistance-model ranking
assert rows['two-pop (mechanistic)'][0] > rows['Claret (phenom.)'][0]   # week-8: two-pop wins
assert rows['two-pop (mechanistic)'][1] < rows['Claret (phenom.)'][1]   # k_g: Claret wins
# Inversion 2: complete responder
assert rows['Norton-Simon (CR)'][0] < rows['Claret (phenom.)'][0]       # week-8: undervalued
assert rows['Norton-Simon (CR)'][1] > rows['Claret (phenom.)'][1]       # k_g: rewarded
print('both inversions confirmed')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for name, rid in models.items():
    w = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t)
    g = onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t, survival_link=KG)
    a1.plot(t, w.os_curve, label=f'{name} ({w.median_os:.0f})')
    a2.plot(t, g.os_curve, label=f'{name} ({g.median_os:.0f})')
a1.set(title='week-8 surrogate', xlabel='weeks', ylabel='survival fraction', ylim=(0, 1.02))
a2.set(title='k_g (tail-sensitive)', xlabel='weeks', ylim=(0, 1.02))
a1.legend(fontsize=7); a2.legend(fontsize=7)
plt.show()

## Guardrails: the default is unchanged; the metric is declared, not fitted

Making the metric configurable changes no existing curve (a link with no `link_metric` still reads week-8). A model with no regrowth (k_g undefined) maps to the baseline hazard, never a nan curve. The k_g link's higher recorded C-index does not raise any tier; out-of-context transport still floors to D.

In [ ]:
# Backward compatibility: default == explicit week-8.
d = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, t=t).median_os
e = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, t=t,
                   survival_link='survival_link.nsclc_os_week8').median_os
assert np.isclose(d, e)

# Undefined k_g -> finite, baseline (best) survival.
g = onkos.simulate(ds, 'drug_effect.norton_simon.nsclc', context=ctx, t=t, survival_link=KG)
assert np.all(np.isfinite(g.os_curve))

# Recorded discrimination: k_g out-discriminates early shrinkage (illustrative).
print('C-index  week-8:', ds['survival_link.nsclc_os_week8'].predictive_performance[0].value,
      '| k_g:', ds[KG].predictive_performance[0].value)
print('guardrails OK')